In [0]:
%sql
-- Bronze: prove the padding exists (nice screenshot for PROCESS.md)
SELECT series_id, length(series_id) AS raw_len, length(trim(series_id)) AS trimmed_len
FROM rearc_quest.bronze.bronze_pr_data LIMIT 5;
-- Overlap between the two data files, and what Silver did about it
SELECT source_file, count(*) FROM rearc_quest.silver.silver_pr_observations GROUP BY 1;
SELECT count(*) AS bronze_rows FROM rearc_quest.bronze.bronze_pr_data;
SELECT count(*) AS silver_rows, count(DISTINCT series_id, year, period) AS distinct_keys
FROM rearc_quest.silver.silver_pr_observations; -- the two numbers must match
-- Every series in the fact table has a label
SELECT count(*) AS unlabelled
FROM rearc_quest.silver.silver_pr_observations o
LEFT ANTI JOIN rearc_quest.silver.silver_pr_series_dim d USING (series_id);
-- Series that only have annual averages (no quarters): they will be absent from best-year
SELECT count(DISTINCT series_id) AS annual_only_series
FROM rearc_quest.silver.silver_pr_observations o
WHERE NOT EXISTS (SELECT 1 FROM rearc_quest.silver.silver_pr_observations q
WHERE q.series_id = o.series_id AND q.period <> 'Q05');
-- The three answers
SELECT * FROM rearc_quest.gold.gold_population_stats;
SELECT * FROM rearc_quest.gold.gold_series_best_year ORDER BY series_id LIMIT 20;
SELECT * FROM rearc_quest.gold.gold_prs30006032_q01_population ORDER BY year;
-- Expectation metrics from the pipeline event log (replace with your pipeline id or use the UI)
SELECT timestamp, details:flow_progress.data_quality.expectations
FROM event_log(TABLE(rearc_quest.gold.gold_series_best_year))
WHERE event_type = 'flow_progress' AND details:flow_progress.data_quality IS NOT NULL
ORDER BY timestamp DESC LIMIT 20;